# parameter-wrap-around-tensor — faded example 3: complete the fake optimizer filter

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `parameter-wrap-around-tensor`. The last cell reports your progress on the `Backprop: Parameter wrap around Tensor` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Parameter wrap around Tensor` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`parameter-wrap-around-tensor`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "parameter-wrap-around-tensor"
DD_SUBTOPIC = "Backprop: Parameter wrap around Tensor"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A fake optimizer step models any autograd helper: it filters to MiniTensors via isinstance, then updates each survivor in place. The isinstance gate is exactly what silently drops HAS-A params.

## Faded exercise 3

Complete `fake_optimizer_step` so it only updates items passing the MiniTensor isinstance gate. Fill in the type check.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import numpy as np

class MiniTensor:
    def __init__(self, array, requires_grad=False):
        self.array = np.asarray(array, dtype=float)
        self.requires_grad = requires_grad

class WrapParam:
    def __init__(self, tensor):
        self.tensor = tensor
        self.requires_grad = True

class IsAParam(MiniTensor):
    def __init__(self, array):
        super().__init__(array, requires_grad=True)

def fake_optimizer_step(params, lr):
    count = 0
    for p in params:
        ok = None  # TODO: fill in this step — read the prompt cell above
        if ok:
            p.array -= lr * np.ones_like(p.array)
            count += 1
    return count

print(fake_optimizer_step([WrapParam(np.array([1.0])), IsAParam([2.0])], 0.1))


def _test():
    isa = IsAParam([2.0, 2.0])
    wrap = WrapParam(np.array([1.0]))
    before = isa.array.copy()
    count = fake_optimizer_step([wrap, isa], 0.5)
    # only the IS-A param survives the gate
    assert count == 1
    assert np.allclose(isa.array, before - 0.5)
    # WrapParam was untouched (its .tensor unchanged)
    assert np.allclose(wrap.tensor, [1.0])


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import numpy as np

class MiniTensor:
    def __init__(self, array, requires_grad=False):
        self.array = np.asarray(array, dtype=float)
        self.requires_grad = requires_grad

class WrapParam:
    def __init__(self, tensor):
        self.tensor = tensor
        self.requires_grad = True

class IsAParam(MiniTensor):
    def __init__(self, array):
        super().__init__(array, requires_grad=True)

def fake_optimizer_step(params, lr):
    count = 0
    for p in params:
        ok = isinstance(p, MiniTensor)
        if ok:
            p.array -= lr * np.ones_like(p.array)
            count += 1
    return count

print(fake_optimizer_step([WrapParam(np.array([1.0])), IsAParam([2.0])], 0.1))
```
</details>